# RUNG-26b — RFdiffusion binder, DESIGN-ONLY (loads the already-folded target; NO Boltz)

Root-cause fix after the numpy-ABI churn: the fold notebook needed numpy<2.0 (Boltz) but the design needs
numpy≥2.0 (jax) — switching numpy mid-run kept poisoning the env. **Boltz already folded your target**, so this
notebook drops Boltz entirely: it **loads `pmhc_mut.pdb`/`pmhc_wt.pdb` from Drive** and runs only
RFdiffusion → ProteinMPNN → AF2. numpy + jax stay at Colab's default 2.x (the proven-good ColabDesign config) —
the ONLY pin is the torch family (2.4, for dgl). No numpy switch → no ABI break.

**Prereq:** you've already run the fold notebook through `[CELL 2] OK` (meta.json + PDBs on Drive). ✅ done.
Fresh runtime ▸ GPU(T4). Cell 1 (load target) → 2 install (~12 min) → 3 SMOKE (paste me) → 4 batch → 5 rank.

In [ ]:
#@title Cell 1 — clone + GPU + load the already-folded target (no Boltz)
import os, json
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
META='/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'
assert os.path.exists(META), 'NO meta.json — run the FOLD notebook (binder_rfdiffusion_colab) Cells 1-2 first to fold the target'
M=json.load(open(META))
assert os.path.exists(M['mut_pdb']) and os.path.exists(M['wt_pdb']), 'target PDBs missing on Drive'
print('target:', M['target_id'], '| hotspot B'+str(M['hotspot']), '| mut', M['mut_pdb'])
print('[CELL 1] OK — target loaded from Drive')

In [ ]:
#@title Cell 2 — install RFdiffusion + ProteinMPNN + ColabDesign (ONLY torch pinned; numpy/jax = Colab default)  [~12 min]
import os, glob, subprocess
def sh(c): r=subprocess.run(c,shell=True,capture_output=True,text=True); (r.returncode and print('  !',c[:60],'\n',(r.stderr or '')[-400:])); return r.returncode
# the ONLY version change: torch family -> 2.4 (dgl needs <=2.4). numpy + jax stay Colab-default (proven-good for ColabDesign).
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')
if not os.path.exists('/content/RFdiffusion'):
    sh('apt-get -qq install -y aria2 >/dev/null')
    sh('git clone -q https://github.com/sokrypton/RFdiffusion.git /content/RFdiffusion')
    sh('git clone -q https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN')
    sh('pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator')
    sh('pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger')
    sh('pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html')   # --no-deps: don't bump torch
    sh('pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx')
    sh('cd /content/RFdiffusion/env/SE3Transformer && pip install -q --no-dependencies .')
    sh('pip install -q git+https://github.com/sokrypton/ColabDesign.git@v1.1.1')   # WITH deps (RUNG-26 proven, uses Colab jax)
    os.makedirs('/content/RFdiffusion/models', exist_ok=True)
    sh('cd /content/RFdiffusion/models && aria2c -q -x16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt')
if not glob.glob('params/params_model_*.npz'):
    sh('mkdir -p params && cd params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar) && tar -xf alphafold_params_2022-12-06.tar && rm -f alphafold_params_2022-12-06.tar')
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')   # re-assert torch (ColabDesign deps may bump)
RI=glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py')
print('run_inference.py:', RI[0] if RI else 'NOT FOUND')
print('Complex ckpt:', bool(glob.glob('/content/RFdiffusion/models/Complex_base_ckpt.pt')))
import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())
# prove the import that kept breaking now works (no numpy switch):
from colabdesign import mk_afdesign_model; print('colabdesign import: OK')
print('[CELL 2] done')

In [ ]:
#@title Cell 3 — SMOKE: 1 design (RFdiffusion -> ProteinMPNN -> AF2 vs MUT+WT)  [~6-10 min, GPU]
import os, glob, subprocess, json
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
WORK=M['work']; MUT_PDB=M['mut_pdb']; WT_PDB=M['wt_pdb']; HOT=f"B{M['hotspot']}"; GL=M['groove_len']; PL=M['pep_len']
RI=(glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py'))[0]
CONTIG=f"[A1-{GL}/0 B1-{PL}/0 70-90]"
import gemmi
def binder_chain(pdb):
    st=gemmi.read_structure(pdb)
    for c in st[0]:
        n=len(c)
        if n!=GL and n!=PL and 50<=n<=120: return c.name
    return [c.name for c in st[0]][-1]
def rfdiffuse(prefix, n):
    cmd=(f"cd /content/RFdiffusion && python {RI} inference.input_pdb={MUT_PDB} 'contigmap.contigs={CONTIG}' "
         f"'ppi.hotspot_res=[{HOT}]' inference.output_prefix={prefix} inference.num_designs={n} "
         f"inference.ckpt_override_path=models/Complex_base_ckpt.pt")
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True); print('  rf:',(r.stderr or '')[-700:]); return r
def mpnn(bb):
    ch=binder_chain(bb); out=f'{WORK}/mpnn'; os.makedirs(out, exist_ok=True)
    subprocess.run(f"cd /content/ProteinMPNN && python protein_mpnn_run.py --pdb_path {bb} --pdb_path_chains {ch} "
                   f"--out_folder {out} --num_seq_per_target 1 --sampling_temp 0.1 --seed 1 --batch_size 1",
                   shell=True, capture_output=True, text=True)
    fa=sorted(glob.glob(f'{out}/seqs/*.fa'))
    if not fa: return None
    s=[l.strip() for l in open(fa[-1]) if l.strip() and not l.startswith('>')][-1]
    return s.split('/')[0] if '/' in s else s
from colabdesign import mk_afdesign_model, clear_mem
def score(target_pdb, seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder', use_multimer=False, num_recycles=3, data_dir='.')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=len(seq), hotspot=HOT, rm_target_seq=False, ignore_missing=True)
    m.predict(seq=seq, verbose=False); log=m.aux['log']
    return {'pae_interaction':round(float(log['i_pae'])*31.0,3),'binder_plddt':round(float(log['plddt'])*100.0,1)}
rfdiffuse(f'{WORK}/smoke/d', 1)
bb=sorted(glob.glob(f'{WORK}/smoke/*.pdb'))
print('backbone:', bb[-1] if bb else 'NONE — paste rf stderr above')
if bb:
    seq=mpnn(bb[-1]); print('binder seq:', seq)
    if seq:
        mut=score(MUT_PDB,seq); wt=score(WT_PDB,seq)
        print('MUT',mut,'| WT',wt,'| disc(Å)',round(wt['pae_interaction']-mut['pae_interaction'],2))
        print('[CELL 3] SMOKE OK — launch Cell 4')

In [ ]:
#@title Cell 4 — THE BATCH (time-boxed + resumable)  [~4 h, GPU]
max_hours=4.0 #@param {type:'number'}
import os, glob, time, json
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); WORK=M['work']
DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True); PAE_BIND=10.0; PLDDT_MIN=80.0
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/metrics.json'))
print(f'[CELL 4] resume from {i}; until', time.strftime('%H:%M',time.localtime(t_end)))
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/metrics.json'): i+=1; continue
    os.makedirs(dd, exist_ok=True)
    try:
        rfdiffuse(f'{dd}/bb', 1); bb=sorted(glob.glob(f'{dd}/*.pdb'))
        if not bb: raise RuntimeError('no backbone')
        seq=mpnn(bb[-1])
        if not seq: raise RuntimeError('no mpnn seq')
        mut=score(MUT_PDB,seq); rec={'design_id':did,'target':M['target_id'],'sequence':seq,'mut':mut}
        if mut['pae_interaction']<=PAE_BIND and mut['binder_plddt']>=PLDDT_MIN: rec['wt']=score(WT_PDB,seq)
        json.dump(rec, open(f'{dd}/metrics.json','w'))
        print(f"  {did}: mut_pae={mut['pae_interaction']} plddt={mut['binder_plddt']}"+(f" DISC={round(rec['wt']['pae_interaction']-mut['pae_interaction'],2)}" if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/metrics.json','w'))
    i+=1
print(f'[CELL 4] done — {i} attempted. Run Cell 5.')

In [ ]:
#@title Cell 5 — RANK + bundle
import os, sys, json, glob, zipfile, shutil
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); DES=f"{M['work']}/designs"
dst='runs/rung26b_rfdiff'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m,d)
os.system(f'{sys.executable} scripts/52_binder_design.py rank {dst}')
b='/content/rung26b_rfdiff.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True):
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 5] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')